In [1]:
from openai import OpenAI

In [2]:
client = OpenAI(api_key = "your_subscrition_key") #enter your subscription key

In [3]:
assistant = client.beta.assistants.create(
    name="Research paper Assistant",
    instructions="Use file search to answer user questions for the research papers.",
    model="gpt-4o-mini",
    tools=[{"type": "file_search"}]
)

print(f"Assistant Created: {assistant.id}")

Assistant Created: asst_2pkKkwSb2BW9P5Lr5UyYy6yF


In [4]:
uploaded_file = client.files.create(
    file=open("RLHF.pdf", "rb"),
    purpose="assistants"
)

print(f"File Uploaded: {uploaded_file.id}")

vector_store = client.beta.vector_stores.create(name="PDFVectorStore")

client.beta.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=uploaded_file.id
)

print(f"File Added to Vector Store: {vector_store.id}")


File Uploaded: file-FiFtFVXiVgRtkTH9XZaXPy
File Added to Vector Store: vs_6813d7c47fa08191bbf71b7e50b5eb6a


In [14]:
assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

print(assistant)

Assistant(id='asst_2pkKkwSb2BW9P5Lr5UyYy6yF', created_at=1746130880, description=None, instructions='Use file search to answer user questions for the research papers.', metadata={}, model='gpt-4o-mini', name='Research paper Assistant', object='assistant', tools=[FileSearchTool(type='file_search', file_search=FileSearch(max_num_results=None, ranking_options=FileSearchRankingOptions(score_threshold=0.0, ranker='default_2024_08_21')))], response_format='auto', temperature=1.0, tool_resources=ToolResources(code_interpreter=None, file_search=ToolResourcesFileSearch(vector_store_ids=['vs_6813d7c47fa08191bbf71b7e50b5eb6a'])), top_p=1.0, reasoning_effort=None)


In [15]:
thread = client.beta.threads.create()
print(thread.id)

thread_IPMjx2I9DibtKzpu2nRixu8H


In [16]:
query = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="What is the research paper about?"
)

run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id
)

print(thread.id)
print(f"Query Sent: {query.id}")

thread_IPMjx2I9DibtKzpu2nRixu8H
Query Sent: msg_VghToRkyJG5q6AXSjF3xWynN


In [19]:
run = client.beta.threads.runs.retrieve(
  thread_id=thread.id,
  run_id=run.id
)

print(run)

Run(id='run_0yvx36XF4u5GSGtdKuNvrH7L', assistant_id='asst_2pkKkwSb2BW9P5Lr5UyYy6yF', cancelled_at=None, completed_at=None, created_at=1746130927, expires_at=1746131527, failed_at=None, incomplete_details=None, instructions='Use file search to answer user questions for the research papers.', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-4o-mini', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=1746130927, status='in_progress', thread_id='thread_IPMjx2I9DibtKzpu2nRixu8H', tool_choice='auto', tools=[FileSearchTool(type='file_search', file_search=FileSearch(max_num_results=None, ranking_options=FileSearchRankingOptions(score_threshold=0.0, ranker='default_2024_08_21')))], truncation_strategy=TruncationStrategy(type='auto', last_messages=None), usage=None, temperature=1.0, top_p=1.0, tool_resources={}, reasoning_effort=None)


In [21]:
messages = list(client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id))
print(messages)

[Message(id='msg_vPTYPz3gusIFUIoVEzppXtUW', assistant_id='asst_2pkKkwSb2BW9P5Lr5UyYy6yF', attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[FileCitationAnnotation(end_index=636, file_citation=FileCitation(file_id='file-FiFtFVXiVgRtkTH9XZaXPy'), start_index=624, text='【4:0†source】', type='file_citation'), FileCitationAnnotation(end_index=924, file_citation=FileCitation(file_id='file-FiFtFVXiVgRtkTH9XZaXPy'), start_index=912, text='【4:0†source】', type='file_citation'), FileCitationAnnotation(end_index=936, file_citation=FileCitation(file_id='file-FiFtFVXiVgRtkTH9XZaXPy'), start_index=924, text='【4:5†source】', type='file_citation'), FileCitationAnnotation(end_index=1241, file_citation=FileCitation(file_id='file-FiFtFVXiVgRtkTH9XZaXPy'), start_index=1229, text='【4:3†source】', type='file_citation'), FileCitationAnnotation(end_index=1569, file_citation=FileCitation(file_id='file-FiFtFVXiVgRtkTH9XZaXPy'), start_index=1557, text='【4:4†source】', type='file_cita

In [22]:
message_content = messages[0].content[0].text
annotations = message_content.annotations
citations = []
for index, annotation in enumerate(annotations):
    message_content.value = message_content.value.replace(annotation.text, f"[{index}]")
    if file_citation := getattr(annotation, "file_citation", None):
        cited_file = client.files.retrieve(file_citation.file_id)
        citations.append(f"[{index}] {cited_file.filename}")

print(message_content.value)

The research paper titled "Reinforcement Learning from Human Feedback" (RLHF) offers an introduction to the domain of RLHF and its application in training language models. The paper discusses how RLHF is critical in incorporating human preferences into AI systems, making it an essential tool for modern machine learning applications, particularly for large language models (LLMs).

### Key Highlights:
1. **Overview and Purpose:** 
   - The paper aims to introduce the core methods and concepts associated with RLHF, detailing its significance in training AI systems to follow human preferences and improve user interaction[0].

2. **Structure of the Paper:**
   - It starts with a historical context that includes the origins of RLHF, definitions, and several chapters that outline methods for data collection, reward modeling, and optimization strategies involved in RL with respect to human feedback[0][2].

3. **Process of RLHF:**
   - The RLHF process is organized into three key steps:
     1.